# EDA

In [1]:

w_d = '/home/edgar/Dev/uni/university-7th-semester/deep_learning/project2/data/raw/' 

In [2]:
import pandas as pd

name = 'CFG_42_EDS_Data-_1_.csv'
df = pd.read_csv(w_d + name)

/tmp/ipykernel_4326/4187181730.py:4: DtypeWarning: Columns (1: Analogue_Signals.30-NI-3271-A, 2: Analogue_Signals.30-VI-3274-A, 3: Analogue_Signals.30-TI-3273, 4: Analogue_Signals.30-TI-3275) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(w_d + name)


In [3]:
df.shape

(525601, 5)

In [4]:
df.head(5)

,Date,Analogue_Signals.30-NI-3271-A,Analogue_Signals.30-VI-3274-A,Analogue_Signals.30-TI-3273,Analogue_Signals.30-TI-3275
0,NaN,Prod Centr 42 Torque,Prod Centr 42 Vibration,Prod Centr 2 Main Brg Temp - Solid end,Prod Centr 2 Main Brg Temp - Liquid End
1,9/5/2022 1:00,59.98,0.29,58.08,57.09
2,9/5/2022 1:01,59.99,0.35,57.90,57.08
3,9/5/2022 1:02,59.81,0.21,57.82,57.12
4,9/5/2022 1:03,59.63,0.23,57.84,57.12


In [5]:
df.tail(5)

,Date,Analogue_Signals.30-NI-3271-A,Analogue_Signals.30-VI-3274-A,Analogue_Signals.30-TI-3273,Analogue_Signals.30-TI-3275
525596,9/5/2023 0:55,5.547265,5.208514,54.378161,52.595678
525597,9/5/2023 0:56,5.58938,5.395895,54.342607,52.625317
525598,9/5/2023 0:57,5.647873,5.355612,54.364335,52.555122
525599,9/5/2023 0:58,5.638514,5.360494,54.321049,52.56515
525600,9/5/2023 0:59,5.605332,4.950078,54.31169,52.548103


In [6]:
df.dtypes

Date                                str
Analogue_Signals.30-NI-3271-A    object
Analogue_Signals.30-VI-3274-A    object
Analogue_Signals.30-TI-3273      object
Analogue_Signals.30-TI-3275      object
dtype: object

In [7]:
DATE_FORMAT = "%m/%d/%Y %H:%M"

df["Date_parsed"] = pd.to_datetime(
    df["Date"],
    format=DATE_FORMAT,
    errors="coerce", # turns invalid dates into NaT
)

In [8]:
df["Date_parsed"].isna().sum()

np.int64(1)

In [9]:
df.loc[df["Date_parsed"].isna()]

,Date,Analogue_Signals.30-NI-3271-A,Analogue_Signals.30-VI-3274-A,Analogue_Signals.30-TI-3273,Analogue_Signals.30-TI-3275,Date_parsed
0,NaN,Prod Centr 42 Torque,Prod Centr 42 Vibration,Prod Centr 2 Main Brg Temp - Solid end,Prod Centr 2 Main Brg Temp - Liquid End,NaT


In [10]:
signal_cols = [
    "Analogue_Signals.30-VI-3274-A",  # vibration
    "Analogue_Signals.30-NI-3271-A",  # torque
    "Analogue_Signals.30-TI-3273",   # bearing temperature A
    "Analogue_Signals.30-TI-3275",   # bearing temperature B
]

valid_rows = df.loc[df["Date_parsed"].notna()].copy()
raw_signals = valid_rows[signal_cols]
numeric_signals = raw_signals.apply(pd.to_numeric, errors="coerce")

pd.DataFrame({
    "blank": raw_signals.isna().sum(),
    "non_numeric": (raw_signals.notna() & numeric_signals.isna()).sum(),
})

,blank,non_numeric
Analogue_Signals.30-VI-3274-A,43094,0
Analogue_Signals.30-NI-3271-A,43094,0
Analogue_Signals.30-TI-3273,43094,0
Analogue_Signals.30-TI-3275,58704,0


In [11]:
valid_dates = valid_rows["Date_parsed"].sort_values()
print(valid_dates.is_unique)
print(valid_dates.diff().value_counts().head())

True
Date_parsed
0 days 00:01:00    525599
Name: count, dtype: int64


In [12]:
analysis_df = valid_rows.set_index("Date_parsed").sort_index()
monthly_missing_pct = (
    analysis_df[signal_cols]
    .isna()
    .resample("MS")
    .mean()
    .mul(100)
    .round(1)
)
monthly_missing_pct

,Analogue_Signals.30-VI-3274-A,Analogue_Signals.30-NI-3271-A,Analogue_Signals.30-TI-3273,Analogue_Signals.30-TI-3275
Date_parsed,,,,
2022-09-01,0.0,0.0,0.0,0.0
2022-10-01,0.1,0.1,0.1,35.0
2022-11-01,0.0,0.0,0.0,0.0
2022-12-01,0.0,0.0,0.0,0.0
2023-01-01,0.0,0.0,0.0,0.0
2023-02-01,0.0,0.0,0.0,0.0
2023-03-01,0.0,0.0,0.0,0.0
2023-04-01,0.0,0.0,0.0,0.0
2023-05-01,10.0,10.0,10.0,10.0


In [13]:
missing = analysis_df[signal_cols].isna()

pd.Series({
    "minutes with any signal missing": missing.any(axis=1).sum(),
    "minutes with all four signals missing": missing.all(axis=1).sum(),
    "minutes missing only temperature B": (missing[signal_cols[3]] & ~missing[signal_cols[:3]].any(axis=1)).sum(),
})

minutes with any signal missing          58704
minutes with all four signals missing    43094
minutes missing only temperature B       15610
dtype: int64

In [14]:
all_four_missing = missing.all(axis=1)

temp_b_only_missing = (missing[signal_cols[3]] & ~missing[signal_cols[:3]].any(axis=1))
daily_missing = pd.DataFrame({
    "all_four_missing_minutes": all_four_missing.resample("D").sum(),
    "temp_B_only_missing_minutes": temp_b_only_missing.resample("D").sum(),
})

daily_missing.loc[daily_missing.any(axis=1)]

,all_four_missing_minutes,temp_B_only_missing_minutes
Date_parsed,,
2022-10-05,29,0
2022-10-11,1,929
2022-10-12,0,1440
2022-10-13,0,1440
2022-10-14,0,1440
2022-10-15,0,1440
2022-10-16,0,1440
2022-10-17,0,1440
2022-10-18,0,1440


## Initial signal-value summary

The dataset contains 525,600 rows with valid timestamps, sampled once per minute. The CSV also has one row with no valid timestamp; its signal fields contain descriptive labels rather than measurements. I exclude that row from analysis but keep the source CSV unchanged.

| Signal | Numeric readings | Valid readings | Mean | Median | Min | Max | Published alarm threshold |
|---|---:|---:|---:|---:|---:|---:|---|
| Vibration (mm/s RMS) | 482,506 | 91.8% | 5.36 | 6.16 | 0.05 | 19.54 | 12 mm/s |
| Torque (% nominal) | 482,506 | 91.8% | 70.27 | 76.17 | 0.00 | 94.47 | 85% |
| Bearing temperature A (°C) | 482,506 | 91.8% | 66.17 | 67.24 | 17.24 | 81.51 | 85 °C |
| Bearing temperature B (°C) | 466,896 | 88.8% | 55.50 | 63.76 | 0.00 | 115.48 | 85 °C |

The `count` and valid-reading percentages use the 525,600 valid-timestamp rows as the denominator. After excluding the descriptive row, the sensor columns contain no non-numeric values; the remaining absent readings are blank.

### Initial interpretation

- The observed maxima for vibration, torque, and bearing temperature B exceed their published alarm thresholds. This shows that alarm-range observations are present; it does not tell us how many or whether they are sensor errors.
- Bearing temperature A's observed maximum is below its threshold.
- Torque reaches zero, which may reflect an idle or stopped state. Bearing temperature B also reaches zero; we should inspect those timestamps and the other signals before deciding whether that value is plausible.
- Alarm thresholds are not automatic data-cleaning limits. Keep these readings for now and investigate them before deciding how to handle them.

### Initial data-quality assessment

The dataset is usable for the forecasting project, but it has substantial structured missingness: 58,704 of the 525,600 timestamps have at least one missing signal. Of these, 43,094 minutes are missing all four signals and 15,610 are missing only bearing temperature B. There are long, multi-day gaps, so I will not interpolate across them or treat them as randomly scattered missing values.

For forecasting, I will require complete input windows and an actually observed target vibration reading. The cause of each long gap—machine downtime or a data-collection interruption—cannot be established from these readings alone.
This is not a reason to abandon the dataset. The missingness is a modeling constraint we can handle explicitly, and the data still has many valid observations. It also makes the project more realistic: documenting why you exclude incomplete windows is valuable engineering work.
Next, let’s count how often each signal exceeds its alarm threshold and inspect the timestamps around high vibration and the zero temperature-B readings. That will help us distinguish unusual-but-real operating conditions from values that need further investigation.

In [15]:
summary = numeric_signals.describe().T[
    ["count", "mean", "std", "min", "25%", "50%", "75%", "max"]
].round(2)
summary

,count,mean,std,min,25%,50%,75%,max
Analogue_Signals.30-VI-3274-A,482506.0,5.36,2.91,0.05,4.48,6.16,7.55,19.54
Analogue_Signals.30-NI-3271-A,482506.0,70.27,20.09,0.00,65.68,76.17,82.61,94.47
Analogue_Signals.30-TI-3273,482506.0,66.17,7.13,17.24,61.99,67.24,71.32,81.51
Analogue_Signals.30-TI-3275,466896.0,55.50,22.67,0.00,57.82,63.76,68.40,115.48


In [16]:
numeric_by_time = analysis_df[signal_cols].apply(pd.to_numeric, errors='coerce')

ALARM_LIMITS = {
    "vibration_mm_s_rms": (signal_cols[0], 12.0),
    "torque_percent": (signal_cols[1], 85.0),
    "bearing_temp_a_c": (signal_cols[2], 85.0),
    "bearing_temp_b_c": (signal_cols[3], 85.0),
}

alarm_rows = []

for signal_name, (column, threshold) in ALARM_LIMITS.items():
    readings = numeric_by_time[column]
    observed_count = int(readings.count())
    exceedance_count = int(readings.ge(threshold).sum())
    alarm_rows.append({
        "signal": signal_name,
        "threshold": threshold,
        "observed_readings": observed_count,
        "readings_at_or_above_threshold": exceedance_count,
        "percent_of_observed": round(100*exceedance_count/observed_count,2),
    })

alarm_summary = pd.DataFrame(alarm_rows).set_index("signal")
alarm_summary

,threshold,observed_readings,readings_at_or_above_threshold,percent_of_observed
signal,,,,
vibration_mm_s_rms,12.0,482506,394,0.08
torque_percent,85.0,482506,79250,16.42
bearing_temp_a_c,85.0,482506,0,0.00
bearing_temp_b_c,85.0,466896,7,0.00


In [17]:
# Get the vibration column and its published threshold
vibration_column = signal_cols[0]
vibration_threshold = 12.0

# Keep rows where vibration meets or exceeds the threshold
high_vibration_rows = numeric_by_time.loc[numeric_by_time[vibration_column].ge(vibration_threshold)]

# Show the then highest vibration readings with the other sensor values
high_vibration_rows.sort_values(
    vibration_column,
    ascending=False,
).head(10)


,Analogue_Signals.30-VI-3274-A,Analogue_Signals.30-NI-3271-A,Analogue_Signals.30-TI-3273,Analogue_Signals.30-TI-3275
Date_parsed,,,,
2023-07-25 21:12:00,19.542737,8.732738,61.151647,59.565283
2023-08-27 13:06:00,18.725872,6.515602,58.628178,57.495995
2023-04-20 05:24:00,18.635843,5.993134,63.223011,57.183142
2023-08-27 13:05:00,18.276585,6.536660,58.732281,57.630396
2023-08-27 13:07:00,17.527122,6.424811,58.626971,57.592517
2023-07-04 22:24:00,17.000992,7.709163,64.609749,57.504016
2023-04-20 05:25:00,16.854526,6.249790,63.128754,57.168342
2023-08-26 06:41:00,16.798779,6.392615,52.938243,57.206806
2023-05-14 06:15:00,15.776364,7.524224,69.862671,63.108995


### High-vibration readings

The ten largest vibration readings are between approximately 15.7 and 19.5 mm/s RMS. Several occur in short clusters, including four consecutive high readings on August 27. At those timestamps, measured torque is low (about 6–7% of nominal) and bearing temperatures are moderate. Thus, these peaks are not obviously associated with high torque or high temperature in the same minute.

This is only an initial observation. A high reading does not establish its cause or prove a sensor error. The threshold count measures minutes, not separate events; neighboring readings must be inspected to understand each excursion.

In [18]:
# select ten minutes before through ten minutes after the 13:06 peak in august 28
context = numeric_by_time.loc[
    "2023-08-27 12:56:00":"2023-08-27 13:16:00",
    signal_cols,
]

context

,Analogue_Signals.30-VI-3274-A,Analogue_Signals.30-NI-3271-A,Analogue_Signals.30-TI-3273,Analogue_Signals.30-TI-3275
Date_parsed,,,,
2023-08-27 12:56:00,0.263472,3.911723,59.727372,58.329469
2023-08-27 12:57:00,0.244022,3.936791,59.584648,58.233376
2023-08-27 12:58:00,0.263472,3.915123,59.516797,58.182880
2023-08-27 12:59:00,0.219406,3.977569,59.413851,58.068756
2023-08-27 13:00:00,0.290125,3.974698,59.329619,58.045929
2023-08-27 13:01:00,0.292465,3.967041,59.241744,58.009430
2023-08-27 13:02:00,0.273482,3.978616,59.098366,57.899933
2023-08-27 13:03:00,0.289189,3.880179,58.983902,57.854657
2023-08-27 13:04:00,0.341917,1.457600,58.817279,57.785218


In [19]:
# Count high-vibration minute readings for each day
daily_high_vibration = high_vibration_rows.resample("D").size()

# Keep only days with at least one reading at or above the threshold
daily_high_vibration = daily_high_vibration[daily_high_vibration.gt(0)]

# show the days with the most high-vibration readings first
daily_high_vibration.sort_values(ascending=False)


Date_parsed
2023-03-15    262
2023-04-30     65
2023-07-23     16
2023-08-17     12
2023-08-27      6
2023-07-24      5
2023-03-09      5
2023-08-26      4
2023-07-03      3
2023-05-14      3
2023-07-21      2
2023-04-20      2
2023-08-23      2
2023-04-04      1
2023-07-05      1
2023-07-04      1
2023-07-11      1
2023-07-08      1
2023-08-06      1
2023-07-25      1
dtype: int64

In [20]:
# select the vibration time series
vibration = numeric_by_time[signal_cols[0]]

# mark each minute wehre vibration meets or exceeds the 12mm/s threshold
above_limit = vibration.ge(12.0)

# compare each minute with the previous one, increment the ID 
# whenever the above/below 
event_id = above_limit.ne(above_limit.shift(fill_value=False)).cumsum()

# Build a temporary table containing timestamps, vibraiton, and event IDs
event_rows = pd.DataFrame({
    "timestamp": numeric_by_time.index,
    "vibration": vibration.to_numpy(),
    "above_limit": above_limit.to_numpy(),
    "event_id": event_id.to_numpy(),
})

# keep only abote threshold minutes and sumarize each consecutive run
event_summary = (
    event_rows.loc[event_rows["above_limit"]]
    .groupby("event_id")
    .agg(
        start=("timestamp", "min"),
        end=("timestamp", "max"),
        minutes=("timestamp", "size"),
        peak_mm_s=("vibration", "max"),
    )
    .sort_values("start")
)

event_summary

,start,end,minutes,peak_mm_s
event_id,,,,
1,2023-03-09 05:17:00,2023-03-09 05:21:00,5,14.099947
3,2023-03-15 15:52:00,2023-03-15 15:52:00,1,12.023438
5,2023-03-15 15:54:00,2023-03-15 15:55:00,2,12.106569
7,2023-03-15 15:57:00,2023-03-15 15:57:00,1,12.401007
9,2023-03-15 15:59:00,2023-03-15 19:18:00,200,13.330037
11,2023-03-15 19:20:00,2023-03-15 19:41:00,22,12.599862
13,2023-03-15 19:43:00,2023-03-15 19:43:00,1,12.402940
15,2023-03-15 19:45:00,2023-03-15 19:46:00,2,12.227054
17,2023-03-15 19:50:00,2023-03-15 19:52:00,3,12.439460


In [21]:
event_summary["minutes"].sum()

np.int64(394)

In [22]:
# Temperature B is the fourth signal in our agreed column order.
temperature_b_column = signal_cols[3]

# Select its time series; the datetime index is preserved.
temperature_b_readings = numeric_by_time[temperature_b_column]

# Create True/False flags for readings at or above 85 °C.
temperature_b_at_alarm_limit = temperature_b_readings.ge(85.0)

# Create True/False flags for readings exactly equal to 0 °C.
temperature_b_is_zero = temperature_b_readings.eq(0.0)

# Summarize observed readings, high readings, and zero readings.
temperature_b_counts = pd.Series({
    "observed_readings": temperature_b_readings.count(),
    "readings_at_or_above_85C": temperature_b_at_alarm_limit.sum(),
    "readings_equal_to_zero": temperature_b_is_zero.sum(),
})

# Display the summary.
temperature_b_counts

observed_readings           466896
readings_at_or_above_85C         7
readings_equal_to_zero       60393
dtype: int64

In [23]:
# Select timestamps where temperature B reached or exceeded 85 °C.
# Show all four signals so we can compare their readings at those timestamps.
temperature_b_alarm_context = numeric_by_time.loc[
    temperature_b_at_alarm_limit,
    signal_cols,
]

# Display the selected rows, retaining their timestamps as the index.
temperature_b_alarm_context

,Analogue_Signals.30-VI-3274-A,Analogue_Signals.30-NI-3271-A,Analogue_Signals.30-TI-3273,Analogue_Signals.30-TI-3275
Date_parsed,,,,
2023-07-17 16:43:00,11.335978,7.646868,81.470928,91.346127
2023-07-17 16:46:00,11.315053,7.600366,81.369347,90.666667
2023-07-30 13:42:00,5.746242,70.001982,62.940632,93.698630
2023-07-30 13:43:00,5.488670,70.301369,63.103683,107.788650
2023-07-30 13:49:00,0.829786,4.515144,63.279163,115.477019
2023-07-30 13:50:00,0.718242,4.220169,63.261616,88.339133
2023-07-30 13:54:00,0.790570,4.174105,62.843803,99.181048


In [24]:
# Select timestamps where temperature B was exactly zero.
temperature_b_zero_context = numeric_by_time.loc[
    temperature_b_is_zero,
    signal_cols,
]

# Show up to ten examples with the other three signal values.
temperature_b_zero_context.head(10)

,Analogue_Signals.30-VI-3274-A,Analogue_Signals.30-NI-3271-A,Analogue_Signals.30-TI-3273,Analogue_Signals.30-TI-3275
Date_parsed,,,,
2022-10-22 04:41:00,0.314031,54.984970,57.664989,0.0
2022-10-22 04:42:00,0.349676,55.099775,57.514668,0.0
2022-10-22 04:43:00,0.327703,55.006028,57.392651,0.0
2022-10-22 04:44:00,0.305730,55.198596,57.542076,0.0
2022-10-22 04:45:00,0.373178,55.342945,57.633327,0.0
2022-10-22 04:46:00,0.389685,55.202563,57.506981,0.0
2022-10-22 04:47:00,0.466229,55.041122,57.631656,0.0
2022-10-22 04:48:00,0.488725,54.962691,57.729676,0.0
2022-10-22 04:49:00,0.549490,55.195545,57.605248,0.0


In [25]:
# Count zero-valued temperature-B readings in each calendar day.
daily_temperature_b_zeros = temperature_b_is_zero.resample("D").sum()

# Keep only days that contain at least one zero reading.
daily_temperature_b_zeros = daily_temperature_b_zeros[
    daily_temperature_b_zeros.gt(0)
]

# Display the affected days and their zero-reading counts.
daily_temperature_b_zeros

Date_parsed
2022-10-22    1159
2022-10-23    1440
2022-10-24    1440
2022-10-25    1440
2022-10-26    1440
2022-10-27    1440
2022-10-28    1440
2022-10-29    1440
2022-10-30    1440
2022-10-31    1440
2022-11-01    1428
2022-11-02     798
2022-11-03    1165
2022-11-04    1440
2022-11-05    1440
2022-11-06    1440
2022-11-07    1440
2022-11-08    1440
2022-11-09    1440
2022-11-10    1440
2022-11-11    1440
2022-11-12    1440
2022-11-13    1440
2022-11-14    1440
2022-11-15    1440
2022-11-16    1440
2022-11-17    1440
2022-11-18    1440
2022-11-19    1440
2022-11-20    1440
2022-11-21    1440
2022-11-22    1440
2022-11-23    1440
2022-11-24    1440
2022-11-25    1440
2022-11-26    1440
2022-11-27    1440
2022-11-28    1440
2022-11-29    1440
2022-11-30     873
2023-05-21     186
2023-07-02      49
2023-07-17     115
2023-07-27     533
2023-07-28    1440
2023-07-29    1440
2023-07-30     807
Name: Analogue_Signals.30-TI-3275, dtype: int64

In [26]:
# Select the period just before and after temperature B changes
# from blank readings to zero readings.
transition_context = numeric_by_time.loc[
    "2022-10-22 04:31:00":"2022-10-22 04:51:00",
    signal_cols,
]

# Display all four sensor readings in that short window
transition_context

,Analogue_Signals.30-VI-3274-A,Analogue_Signals.30-NI-3271-A,Analogue_Signals.30-TI-3273,Analogue_Signals.30-TI-3275
Date_parsed,,,,
2022-10-22 04:31:00,0.336527,54.382239,58.166935,NaN
2022-10-22 04:32:00,0.303532,54.825526,58.038480,NaN
2022-10-22 04:33:00,0.325639,54.885176,58.219730,NaN
2022-10-22 04:34:00,0.349389,54.855574,58.328526,NaN
2022-10-22 04:35:00,0.315124,54.845501,58.551224,NaN
2022-10-22 04:36:00,0.348760,54.984206,58.367131,NaN
2022-10-22 04:37:00,0.335027,54.773785,58.102692,NaN
2022-10-22 04:38:00,0.350774,55.125352,58.074615,NaN
2022-10-22 04:39:00,0.335027,54.790876,58.128430,NaN


### Temperature B changes from blank to zero

From 04:31 to 04:40 on October 22, temperature B is blank. Starting at 04:41, it is recorded as exactly 0.0 °C, while torque remains around 55% of nominal and bearing temperature A remains around 58 °C. This suggests the zeros may represent a sensor or data-system issue rather than a real bearing temperature. This is a working hypothesis, not a confirmed cause; the raw data remains unchanged.

In [27]:
# Show all four signals from ten minutes before through ten minutes
# after the highest temperature-B reading
temperature_b_peak_context = numeric_by_time.loc[
    "2023-07-30 13:39:00":"2023-07-30 13:59:00",
    signal_cols,
]

# Display the timestamped measurements in that interval
temperature_b_peak_context

,Analogue_Signals.30-VI-3274-A,Analogue_Signals.30-NI-3271-A,Analogue_Signals.30-TI-3273,Analogue_Signals.30-TI-3275
Date_parsed,,,,
2023-07-30 13:39:00,5.945220,70.441750,62.587166,51.428571
2023-07-30 13:40:00,5.867857,69.142441,62.738518,65.518591
2023-07-30 13:41:00,6.096895,71.663994,62.889763,79.608611
2023-07-30 13:42:00,5.746242,70.001982,62.940632,93.698630
2023-07-30 13:43:00,5.488670,70.301369,63.103683,107.788650
2023-07-30 13:44:00,5.132219,44.601512,63.159840,0.373350
2023-07-30 13:45:00,0.751354,6.891737,63.252424,7.840357
2023-07-30 13:46:00,0.794079,5.817502,63.311413,15.307364
2023-07-30 13:47:00,0.758373,5.195697,63.290862,40.668346


### Temperature-B readings around the July 30 peak

From 13:39 to 13:43, temperature B rises from about 51 °C to 108 °C, then falls to about 0.4 °C at 13:44. It later jumps to about 115.5 °C at 13:49. During the same interval, bearing temperature A remains near 63 °C. Torque and vibration also change around 13:44–13:45, indicating a possible operating transition, but the abrupt temperature-B changes remain suspicious.

These readings are evidence of a possible sensor or data-collection issue, not proof of a specific cause. The raw values are preserved; we will decide how to handle them only after checking other zero-value periods and the dataset documentation.

In [28]:
# Limit the analysis to the four dates with many temperature-B zeros.
july_window = numeric_by_time.loc[
    "2023-07-27":"2023-07-30",
    signal_cols,
]

# Keep the minutes where temperature B is exactly zero.
july_zero_rows = july_window.loc[
    july_window[signal_cols[3]].eq(0.0)
]

# Show the first five zero readings, with all four sensor values.
july_zero_rows.head(5)

,Analogue_Signals.30-VI-3274-A,Analogue_Signals.30-NI-3271-A,Analogue_Signals.30-TI-3273,Analogue_Signals.30-TI-3275
Date_parsed,,,,
2023-07-27 15:01:00,5.181231,64.680249,67.340936,0.0
2023-07-27 15:02:00,5.418662,61.532615,67.362766,0.0
2023-07-27 15:03:00,5.253986,59.487298,67.363497,0.0
2023-07-27 15:04:00,5.366720,59.359428,67.366647,0.0
2023-07-27 15:05:00,5.304524,59.406731,67.361370,0.0


In [29]:
# Show the last five zero readings in the July 27–30 interval.
july_zero_rows.tail(5)

,Analogue_Signals.30-VI-3274-A,Analogue_Signals.30-NI-3271-A,Analogue_Signals.30-TI-3273,Analogue_Signals.30-TI-3275
Date_parsed,,,,
2023-07-30 13:22:00,4.934829,64.759898,63.223011,0.0
2023-07-30 13:23:00,5.074235,64.164185,63.173878,0.0
2023-07-30 13:24:00,4.845960,64.879227,63.086136,0.0
2023-07-30 13:25:00,5.095293,65.037766,63.086136,0.0
2023-07-30 13:26:00,5.179278,64.830093,63.008008,0.0


## Forecast window construction: train and validation

Let `t` be the timestamp of the final input reading. The input is the 30 rows from `t - 29 minutes` through `t`, inclusive, in vibration, torque, temperature A, temperature B order. The label is the recorded vibration at the exact timestamp `t + 10 minutes`. The nine timestamps between `t` and `t + 10` are not part of the input or label and are not checked.

The source timeline stays intact, including rows with blank signals. An input is valid only when all four signals are present in all 30 input rows. The target is valid when vibration is present at `t + 10`; other signals at the target timestamp are not required. Exact zero values for temperature B are masked only in a derived feature table. Values at or above 85 °C are retained and flagged, not removed.

Splits use the source clock (UTC+3): training ends before May 18, 2023 00:00; validation runs from May 18, 2023 00:00 up to but not including July 12, 2023 00:00. A window is accepted only if its entire input and its target both fall inside the same split. We do not calculate or inspect test windows in this section. The preliminary test baseline and some test-period data have already been inspected, so test results must not guide model selection.

In [30]:
# The final input reading is t; include 30 readings ending at t.
WINDOW_MINUTES = 30
FORECAST_HORIZON_MINUTES = 10

# The agreed feature order is vibration, torque, temperature A, temperature B.
VIBRATION_COLUMN = signal_cols[0]
TEMPERATURE_B_COLUMN = signal_cols[3]

# Use half-open split intervals in the CSV's source clock.
VALIDATION_START = pd.Timestamp("2023-05-18 00:00:00")
VALIDATION_END = pd.Timestamp("2023-07-12 00:00:00")

# Restrict this step to data before the test period.
train_validation_rows = numeric_by_time.loc[
    numeric_by_time.index < VALIDATION_END, signal_cols
].copy()

# Keep the unmasked values for the 'before' count and make a separate feature view.
features_before_zero_mask = train_validation_rows.copy()
model_features = train_validation_rows.copy()
temperature_b_zero_mask = model_features[TEMPERATURE_B_COLUMN].eq(0.0)
model_features.loc[temperature_b_zero_mask, TEMPERATURE_B_COLUMN] = float("nan")

# Flag high temperature-B readings for review without excluding or changing them.
temperature_b_alarm_flag = train_validation_rows[TEMPERATURE_B_COLUMN].ge(85.0)
assert model_features.loc[temperature_b_alarm_flag, TEMPERATURE_B_COLUMN].notna().all()

# A row is complete only when every required input signal is present.
raw_rows_complete = features_before_zero_mask[signal_cols].notna().all(axis=1)
model_rows_complete = model_features[signal_cols].notna().all(axis=1)

# The rolling sum is true only when all 30 consecutive minute rows are complete.
raw_input_complete = (
    raw_rows_complete.astype("int8")
    .rolling(WINDOW_MINUTES, min_periods=WINDOW_MINUTES)
    .sum()
    .eq(WINDOW_MINUTES)
)
masked_input_complete = (
    model_rows_complete.astype("int8")
    .rolling(WINDOW_MINUTES, min_periods=WINDOW_MINUTES)
    .sum()
    .eq(WINDOW_MINUTES)
)

# For every possible final input timestamp t, calculate the input start and target time.
anchor_times = train_validation_rows.index
input_start_times = anchor_times - pd.Timedelta(minutes=WINDOW_MINUTES - 1)
target_times = anchor_times + pd.Timedelta(minutes=FORECAST_HORIZON_MINUTES)

# Accept only windows fully inside training or validation. No test mask is created.
train_window = pd.Series(
    (input_start_times >= anchor_times.min()) & (target_times < VALIDATION_START),
    index=anchor_times,
)
validation_window = pd.Series(
    (input_start_times >= VALIDATION_START) & (target_times < VALIDATION_END),
    index=anchor_times,
)
train_or_validation_window = train_window | validation_window

# Look up labels at the exact target timestamps for train/validation anchors only.
target_readings = train_validation_rows[VIBRATION_COLUMN].reindex(
    target_times[train_or_validation_window]
)
target_observed = pd.Series(False, index=anchor_times)
target_observed.loc[train_or_validation_window] = target_readings.notna().to_numpy()

# Combine input completeness, observed vibration labels, and split membership.
train_valid_before_zero_mask = raw_input_complete & target_observed & train_window
validation_valid_before_zero_mask = raw_input_complete & target_observed & validation_window
train_valid_after_zero_mask = masked_input_complete & target_observed & train_window
validation_valid_after_zero_mask = masked_input_complete & target_observed & validation_window

# Store only the timestamps that identify valid train and validation windows.
train_window_anchors = anchor_times[train_valid_after_zero_mask]
validation_window_anchors = anchor_times[validation_valid_after_zero_mask]

In [31]:
# Count accepted anchor timestamps before and after masking temperature-B zeros.
window_counts = pd.DataFrame(
    {
        "before_temperature_b_zero_mask": [
            int(train_valid_before_zero_mask.sum()),
            int(validation_valid_before_zero_mask.sum()),
        ],
        "after_temperature_b_zero_mask": [
            int(train_valid_after_zero_mask.sum()),
            int(validation_valid_after_zero_mask.sum()),
        ],
    },
    index=["train", "validation"],
)

# The loss is the number of otherwise-valid windows affected by a zero in input temperature B.
window_counts["windows_removed_by_temperature_b_zero_mask"] = (
    window_counts["before_temperature_b_zero_mask"]
    - window_counts["after_temperature_b_zero_mask"]
)

# Compare against the independent counts supplied for this checkpoint.
assert window_counts.loc["train", "before_temperature_b_zero_mask"] == 346_879
assert window_counts.loc["validation", "before_temperature_b_zero_mask"] == 40_534
assert window_counts.loc["train", "after_temperature_b_zero_mask"] == 290_543
assert window_counts.loc["validation", "after_temperature_b_zero_mask"] == 40_241

window_counts

,before_temperature_b_zero_mask,after_temperature_b_zero_mask,windows_removed_by_temperature_b_zero_mask
train,346879,290543,56336
validation,40534,40241,293


In [32]:
# The known example ends at 14:00 on May 18, which is in validation.
example_anchor = pd.Timestamp("2023-05-18 14:00:00")
example_input_start = example_anchor - pd.Timedelta(minutes=WINDOW_MINUTES - 1)
example_target_time = example_anchor + pd.Timedelta(minutes=FORECAST_HORIZON_MINUTES)

# Slice input rows inclusively from 13:31 through 14:00 in the required feature order.
example_x = model_features.loc[example_input_start:example_anchor, signal_cols]

# The label is only vibration, looked up at the exact 14:10 timestamp.
example_y = train_validation_rows.at[example_target_time, VIBRATION_COLUMN]

# Verify the shape, consecutive minute timestamps, complete input, and validation membership.
expected_example_times = pd.date_range(
    start=example_input_start, end=example_anchor, freq="min"
)
assert list(example_x.index) == list(expected_example_times)
assert example_x.shape == (30, 4)
assert example_x.columns.tolist() == signal_cols
assert example_x.notna().all().all()
assert pd.notna(example_y)
assert bool(validation_valid_after_zero_mask.loc[example_anchor])
assert example_anchor in validation_window_anchors
assert round(float(example_y), 2) == 5.62

# Show the timestamps, shape, and recorded (unrounded) target.
pd.Series({
    "first_input_timestamp": example_x.index[0],
    "last_input_timestamp": example_x.index[-1],
    "input_shape": str(example_x.shape),
    "target_timestamp": example_target_time,
    "target_vibration_mm_s_raw": float(example_y),
    "target_vibration_mm_s_rounded": round(float(example_y), 2),
})

first_input_timestamp            2023-05-18 13:31:00
last_input_timestamp             2023-05-18 14:00:00
input_shape                                  (30, 4)
target_timestamp                 2023-05-18 14:10:00
target_vibration_mm_s_raw                   5.616358
target_vibration_mm_s_rounded                   5.62
dtype: object

In [33]:
# This training candidate ends just after the May 6–9 all-signal gap.
missing_input_anchor = pd.Timestamp("2023-05-09 11:20:00")
missing_input_start = missing_input_anchor - pd.Timedelta(minutes=WINDOW_MINUTES - 1)
missing_input_target_time = missing_input_anchor + pd.Timedelta(minutes=FORECAST_HORIZON_MINUTES)
missing_input_x = model_features.loc[missing_input_start:missing_input_anchor, signal_cols]
missing_input_y = train_validation_rows.at[missing_input_target_time, VIBRATION_COLUMN]
missing_input_minutes = missing_input_x.index[missing_input_x.isna().any(axis=1)]

# The input is rejected because at least one of its 30 rows lacks a required signal.
assert len(missing_input_x) == WINDOW_MINUTES
assert len(missing_input_minutes) > 0
assert pd.notna(missing_input_y)
assert bool(train_window.loc[missing_input_anchor])
assert not bool(train_valid_after_zero_mask.loc[missing_input_anchor])

pd.Series({
    "input_start": missing_input_start,
    "last_input": missing_input_anchor,
    "input_rows": len(missing_input_x),
    "minutes_with_a_missing_input_signal": len(missing_input_minutes),
    "target_time": missing_input_target_time,
    "target_vibration_observed": pd.notna(missing_input_y),
    "accepted_as_training_window": bool(train_valid_after_zero_mask.loc[missing_input_anchor]),
})

input_start                            2023-05-09 10:51:00
last_input                             2023-05-09 11:20:00
input_rows                                              30
minutes_with_a_missing_input_signal                     15
target_time                            2023-05-09 11:30:00
target_vibration_observed                             True
accepted_as_training_window                          False
dtype: object

In [34]:
# The 30 input rows are before the training boundary, but this target lands exactly at it.
boundary_anchor = pd.Timestamp("2023-05-17 23:50:00")
boundary_input_start = boundary_anchor - pd.Timedelta(minutes=WINDOW_MINUTES - 1)
boundary_target_time = boundary_anchor + pd.Timedelta(minutes=FORECAST_HORIZON_MINUTES)
boundary_x = model_features.loc[boundary_input_start:boundary_anchor, signal_cols]
boundary_y = train_validation_rows.at[boundary_target_time, VIBRATION_COLUMN]

# Both the input and label exist, but no split contains the entire window.
assert len(boundary_x) == WINDOW_MINUTES
assert boundary_x.notna().all().all()
assert pd.notna(boundary_y)
assert boundary_target_time == VALIDATION_START
assert not bool(train_window.loc[boundary_anchor])
assert not bool(validation_window.loc[boundary_anchor])
assert not bool((train_valid_after_zero_mask | validation_valid_after_zero_mask).loc[boundary_anchor])

pd.Series({
    "input_start": boundary_input_start,
    "last_input": boundary_anchor,
    "target_time": boundary_target_time,
    "input_complete": boundary_x.notna().all().all(),
    "target_vibration_observed": pd.notna(boundary_y),
    "target_is_at_validation_boundary": boundary_target_time == VALIDATION_START,
    "accepted_by_train_or_validation": bool((train_valid_after_zero_mask | validation_valid_after_zero_mask).loc[boundary_anchor]),
})

input_start                         2023-05-17 23:21:00
last_input                          2023-05-17 23:50:00
target_time                         2023-05-18 00:00:00
input_complete                                     True
target_vibration_observed                          True
target_is_at_validation_boundary                   True
accepted_by_train_or_validation                   False
dtype: object

### Window-construction checkpoint

The May 18 example verifies the 30 input timestamps, feature order, target timestamp, shape, and recorded target value. The May 9 candidate is rejected because its input contains missing readings even though its target is observed. The May 17 candidate is rejected because its target is exactly at the validation boundary while its input is in training.

The nine minutes between the final input and label are not checked. A temperature-B zero matters only if it occurs in the 30 input rows; temperature B at the target timestamp does not invalidate a vibration label. High temperature-B values remain in the features and are flagged separately. The final test block is not inspected in this section.